# 🎬 CineScore — Phase 2: Feature Engineering

In this phase, we transform the raw cleaned data into a high-signal feature set designed for predictive modeling. The goal is to isolate commercially viable English-language productions and derive metrics that reflect both industry seasonal trends and financial performance.

### 🛠️ Key Transformations

1. **Strict Commercial Filters**: We exclude titles with a budget or revenue below **$100,000**. This ensures our model focuses on "Theatrical Class" releases rather than independent micro-budget experiments or straight-to-DVD titles.

2. **Hollywood MVP Focus**: By filtering for `original_language == 'en'`, we narrow our scope to the English-language market (Hollywood and major global English releases), which often follows specific marketing and ROI patterns.

3. **Seasonality Engineering**: We extract the **Release Month** from the date string. This is critical for capturing "Summer Blockbuster" vs. "October Horror" vs. "December Award" seasonal impacts.

4. **Financial Metrics**: 
   - **Profit**: The raw metric of success ($Revenue - Budget$).
   - **ROI %**: The efficiency metric, calculating the percentage return on every dollar spent.

5. **Genre Distillation**: Movies often carry multiple genres. We extract the **Primary Genre** (first in the list) to categorize the film for simpler one-hot encoding later.

In [5]:
import pandas as pd
import ast
import os

# ── STEP 1: CONFIGURE & INITIALIZE ──────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)

movies_df = pd.DataFrame()
IN_FILE = "tmdb_cleaned_movies.csv"
OUT_FILE = "tmdb_featured_movies.csv"

def get_primary_genre(gs):
    try:
        genres_list = ast.literal_eval(gs)
        return genres_list[0] if isinstance(genres_list, list) and genres_list else 'Unknown'
    except Exception:
        return 'Unknown'

# ── STEP 2: PIPELINE ─────────────────────────────────────────────────────────
if os.path.exists(IN_FILE):
    movies_df = pd.read_csv(IN_FILE)
    print(f"📦 Loaded {len(movies_df):,} records.")

    # Filtering for Commercial English-Language Cinema
    mask = (movies_df['budget'] >= 100_000) & (movies_df['revenue'] >= 100_000) & (movies_df['original_language'] == 'en')
    movies_df = movies_df[mask].copy()
    
    # Feature Extraction
    movies_df['release_date'] = pd.to_datetime(movies_df['release_date'])
    movies_df['release_month'] = movies_df['release_date'].dt.month
    movies_df['profit'] = movies_df['revenue'] - movies_df['budget']
    movies_df['roi_percentage'] = (movies_df['profit'] / movies_df['budget']) * 100
    movies_df['primary_genre'] = movies_df['genre_names'].apply(get_primary_genre)
    
    # Insight Summary
    print(f"✅ Success! Final Cleaned Shape: {movies_df.shape}")
    print("=" * 100)
    print("Top 5 Highest Grossing Blockbusters (Unified View):")
    
    cols = ['title', 'release_month', 'primary_genre', 'budget', 'roi_percentage']
    # Sorted display
    print(movies_df.sort_values(by='revenue', ascending=False)[cols].head(5))
    print("=" * 100)
    
    movies_df.to_csv(OUT_FILE, index=False)
    print(f"💾 Saved output to: {OUT_FILE}")
else:
    print(f"❌ ERROR: Missing '{IN_FILE}'")

📦 Loaded 756,402 records.
✅ Success! Final Cleaned Shape: (334810, 19)
Top 5 Highest Grossing Blockbusters (Unified View):
                               title  release_month    primary_genre       budget  roi_percentage
139945                        Avatar             12           Action  237000000.0     1133.631235
480900             Avengers: Endgame              4        Adventure  356000000.0      686.359298
620407      Avatar: The Way of Water             12  Science Fiction  350000000.0      562.928652
79964                        Titanic             11            Drama  200000000.0     1032.081177
375106  Star Wars: The Force Awakens             12        Adventure  245000000.0      744.172908
💾 Saved output to: tmdb_featured_movies.csv
